> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [モデル 탐색 (Discover)](#モデル-탐색-discover)
- [モデル 비교 및 デプロイ](#モデル-비교-및-デプロイ)
- [Embedding モデル デプロイ](#embedding-モデル-デプロイ)
- [Model Router デプロイ](#model-router-デプロイ)
- [Model Router 構成](#model-router-構成)

## 🎯 学習目標

- モデル 리더보드를 통한 モデル パフォーマンス 비교
- 다양한 AI モデル デプロイ 방법 이해
- Model Router 設定 및 構成
- モデル 라우팅 전략 이해

## ⏱️ 予想所要時間

約15分

## 環境設定

먼저 前へ ノート북에서 作成한 Foundry リソース 情報를 設定합니다.

In [ ]:
# 環境 変数 ロード
import json
import os
import subprocess

# PATH 環境変数 設定 (Azure CLI를 찾을 수 있도록)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# 前へ ノート북에서 保存한 設定 ファイル ロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境 変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境 変数로도 設定 (다른 도구들이 使用할 수 있도록)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定 ファイル '{config_file}'에서 環境 変数를 ロード했습니다.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイル을 찾을 수 없습니다.")
    print("💡 01-setup.ipynb를 먼저 実行하여 環境을 設定하세요.")
    raise

# 必須パッケージのインストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import InteractiveBrowserCredential

print(f"\n💡 使用할 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")

## 使用 가능한 モデル 取得

Azure CLI로 Sweden Central 리전에서 使用 가능한 モデル을 確認합니다.

In [ ]:
# Sweden Central 리전에서 使用 가능한 モデル 取得
print("🔍 Sweden Central 리전에서 使用 가능한 AI モデル 取得 중...")
print("=" * 80)

# Azure CLI로 리전에서 使用 가능한 モデル 取得
# GPT 관련 モデル フィルター링하여 보기 쉽게 表示
import subprocess

cmd = [
    "az", "cognitiveservices", "model", "list",
    "--location", LOCATION,
    "--query", "[?contains(model.name, 'gpt') || contains(model.name, 'embedding')].{Name:model.name, Version:model.version, Format:model.format, Kind:kind}",
    "--output", "table"
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.stderr and not result.stderr.startswith("("):  # Azure CLI warnings 무시
    print(result.stderr)

print("\n" + "=" * 80)
print("💡 전체 モデル リスト을 보려면 아래 명령어를 実行하세요:")
print(f"   !az cognitiveservices model list --location {LOCATION} --output table")
print("\n💡 위 リスト에서 원하는 モデル을 選択하여 아래 셀에서 デプロイ할 수 있습니다.")
print("💡 モデル 名前(Name)과 バージョン(Version)을 確認하세요.")

## GPT-4.1 モデル デプロイ

GPT-4.1 モデル을 デプロイ합니다. GlobalStandard SKU를 使用하여 최적의 パフォーマンス을 제공합니다.

In [ ]:
# GPT-4.1 モデル デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-4.1 \
    --model-name gpt-4.1 \
    --model-format OpenAI \
    --model-version "2025-04-14" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ GPT-4.1 モデル デプロイ 完了!")

## GPT-5.1 モデル デプロイ

GPT-5.1 モデル을 デプロイ합니다. GlobalStandard SKU를 使用하여 최적의 パフォーマンス을 제공합니다.

In [ ]:
# GPT-5.1 モデル デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-5.1 \
    --model-name gpt-5.1 \
    --model-format OpenAI \
    --model-version "2025-11-13" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ GPT-5.1 モデル デプロイ 完了!")

## デプロイ 状態 確認

デプロイ된 モデル의 状態를 確認합니다.

In [ ]:
# デプロイ 状態 確認
!az cognitiveservices account deployment show \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name gpt-4.1 \
    --query "{Name:name, Model:properties.model.name, Version:properties.model.version, Status:properties.provisioningState}" \
    --output table

# 모든 デプロイ된 モデル リスト
print("\n📦 デプロイ된 모든 モデル:")
!az cognitiveservices account deployment list \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --output table

## Embedding モデル デプロイ

Embedding モデル은 텍스트를 벡터로 변환하여 의미적 検索 및 유사도 계산에 使用됩니다.

In [ ]:
# text-embedding-3-large モデル デプロイ (벡터 차원: 3072)
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name text-embedding-3-large \
    --model-name text-embedding-3-large \
    --model-format OpenAI \
    --model-version "1" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ Embedding モデル デプロイ 完了!")

# デプロイ 状態 確認
!az cognitiveservices account deployment show \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name text-embedding-3-large \
    --query "{Name:name, Model:properties.model.name, Status:properties.provisioningState}" \
    --output table
print("   벡터 차원: 3072 (Knowledge Base에서 使用)")

## Model Router デプロイ

Model Router는 여러 モデル 간의 지능형 라우팅을 제공하여 비용, 품질, パフォーマンス을 최적화합니다.

### Routing Mode オプション:

**a) Balanced Mode (균형 모드)** - デフォルト値
- 비용, 품질, パフォーマンス의 균형 유지
- 일반적인 프로덕션 워크ロード에 적합

**b) Quality Mode (품질 모드)**
- 최고 품질의 レスポンス 우선
- 정확도가 重要한 애플리케이션

**c) Cost Mode (비용 모드)**
- 비용 최적화 우선
- 대량의 간단한 リクエスト 처리

In [ ]:
# Model Router デプロイ
!az cognitiveservices account deployment create \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --deployment-name model-router \
    --model-name model-router \
    --model-format OpenAI \
    --model-version "2025-11-18" \
    --sku-name GlobalStandard \
    --sku-capacity 1

print("\n✅ Model Router デプロイ 完了!")
print("\n📊 Model Router 동작 방식:")
print("  使用자 リクエスト → Model Router → 판단:")
print("    - 간단한 질문 → 저비용 モデル")
print("    - 복잡한 분석 → 고품질 モデル")
print("    - 높은 부하 → 부하 분산")

## デプロイ된 モデル 최종 確認

모든 モデル이 정상적으로 デプロイ되었는지 確認합니다.

In [ ]:
# 최종 デプロイ リスト 確認
print("=" * 80)
print("デプロイ된 モデル リスト")
print("=" * 80)

!az cognitiveservices account deployment list \
    --resource-group $RESOURCE_GROUP \
    --name $FOUNDRY_NAME \
    --query "[].{Name:name, Model:properties.model.name, SKU:sku.name, Status:properties.provisioningState}" \
    --output table

print("\n✅ 次へ モデル이 デプロイ되어야 합니다:")
print("  1. gpt-4.1 (Language Model)")
print("  2. gpt-5.1 (Language Model)")
print("  3. text-embedding-3-large (Embedding Model)")
print("  4. model-router (Router)")

print("\n💡 포털 確認: https://ai.azure.com")
print("   Build > Models에서 デプロイ된 モデル을 시각적으로 確認할 수 있습니다.")

## 次のステップ

モデル デプロイ가 完了되었습니다! 이제 이 モデル들을 활용하여 エージェント를 구축해봅시다:

➡️ **[03. エージェント 개발](./03-agents.ipynb)**: 다양한 機能을 가진 AI エージェント를 만들어봅니다.